# Applied Statistics Project
## Population Decline In The West of Ireland
----
Population change is a primary indicator of regional development. It reflects long-term patterns in migration, employment opportunity and housing availability. In Ireland, the _Western Development Commsion_ was set up 
> "in response to public pressure to help tackle the massive population decline in the Western Region" - WesternDevelopment.ie

The focus of the [Western Development Commision's](https://westerndevelopment.ie/) remit - **Clare**, **Galway**, **Mayo**, **Roscommon**, **Sligo**, **Leitrim** and **Donegal**. As an area that I was born in to, emigrated from and returned to raise a family in, I have a vested interest in the region's success and seemed fitting to apply the analyitcal tools and methodologies to this topic. 

Thus far, throughout the course of the Applied Statistics Module, we have looked at -

- [Lady Tasting Tea Experiment](https://en.wikipedia.org/wiki/Lady_tasting_tea)

- [ANOVA (Analysis Of Variance)](https://en.wikipedia.org/wiki/Analysis_of_variance)

- [Normal Distribution](https://en.wikipedia.org/wiki/Normal_distribution)

- Permutations & Combinations

- [Binomial Distribution](https://numpy.org/doc/stable/reference/random/generated/numpy.random.binomial.html)

- Flipping Two & Many Coins

- Bell Curves

- [Histograms with Matplotlib](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)

- [Sampling Distribution](https://en.wikipedia.org/wiki/Sampling_distribution)

The aim of the project is to apply the methodologies from the above to the Central Statistics Office [_Population at Each Census_ dataset](https://data.cso.ie/table/F1001). Specifically, I will quantitatively compare the population dynamics of the WDC counties with the rest of Ireland and assess whether the data reflects a different patterns of growth, decline and/or volatility. 



In [1]:
# Library Imports
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [2]:
# Data Source
df = pd.read_csv('files/populationbycounty1841-2022.csv')

# Printing To Test
df.head()

,Statistic Label,CensusYear,County,Sex,UNIT,VALUE
0,Population at Each Census,1841,State,Both sexes,Number,6528799
1,Population at Each Census,1841,State,Male,Number,3222485
2,Population at Each Census,1841,State,Female,Number,3306314
3,Population at Each Census,1841,Carlow,Both sexes,Number,86228
4,Population at Each Census,1841,Carlow,Male,Number,42428


#### Pivotting The Data
Next step is to pivot this df so that each county has one row per year. The 'Male', 'Female' and 'Both sexes' will get their own column. The *UNIT* and *Statistic Label* columns will be dropped. I will be using the Pandas [*pivot_table*](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html) function to perform this transformation. 

In [3]:
census_pop = df.pivot_table(
    index=["CensusYear","County"], 
    columns="Sex",
    values="VALUE",
    aggfunc="sum"
).reset_index()

census_pop.columns.name=None
census_pop = census_pop.rename_axis(None, axis=1)

# Renaming Column
census_pop.rename(columns={"Both sexes": "FMCombined"}, inplace=True)

# Re-ordering Columns
census_pop = census_pop[["CensusYear", "County", "Female", "Male", "FMCombined"]]

# Printing To Test 
census_pop.head

<bound method NDFrame.head of      CensusYear     County  Female    Male  FMCombined
0          1841     Carlow   43800   42428       86228
1          1841      Cavan  122344  120814      243158
2          1841      Clare  142285  144109      286394
3          1841       Cork  433567  420551      854118
4          1841    Donegal  150627  145821      296448
..          ...        ...     ...     ...         ...
697        2022  Tipperary   84256   83639      167895
698        2022  Waterford   64268   63095      127363
699        2022  Westmeath   48500   47721       96221
700        2022    Wexford   83142   80777      163919
701        2022    Wicklow   79287   76564      155851

[702 rows x 5 columns]>

#### Grouping The Counties
I will add a new column to the dataframe to categorise the counties as either a **'WDC'** county or **'Rest'**. I will use the [*Lambda* function](https://www.w3schools.com/python/python_lambda.asp) to do this.

In [4]:
# Defining WDC Group
wdc = ["Clare", "Galway", "Mayo", "Roscommon", "Sligo", "Leitrim", "Donegal"]

# Adding Group Column With Lambda
census_pop["Group"] = census_pop["County"].apply(lambda x: "WDC" if x in wdc else ("State" if x == "State" else "Rest"))

# Creating Dataframes By Group
df_wdc = census_pop[census_pop["Group"] == "WDC"]
df_rest = census_pop[census_pop["Group"] == "Rest"]
df_state = census_pop[census_pop["Group"] == "State"]

### Starting Point:
I now have the data structured to make a start. I will first visualise the national population trend over time. I will use the [line charts from Plotly](https://plotly.com/python/line-charts/) to begin with. 

In [5]:
# State Population Over Time 
fig = px.line(
    df_state,
    x="CensusYear", 
    y="FMCombined",
    title="Total Population of Rep of Ireland (1841-2022)",
    labels={"FMCombined": "Population", "CensusYear": "Year"},
    markers=True
)
fig.show()


The sharp decline at the start of this trend line cannot be assessed without the context of [The Great Famine](https://en.wikipedia.org/wiki/Great_Famine_(Ireland)) (1845-1852). Emigration levels had been high before and continued for decades after this period. 

> "No country in Europe has been as affected by emigration over the last two centuries as Ireland. Approximately ten million people have emigrated from the island Ireland since 1800." - [UCC Emigre](https://www.ucc.ie/en/emigre/history/#:~:text=No%20country%20in%20Europe%20has,the%20island%20Ireland%20since%201800.)

Outside of the emigration heavily influenced by the _The Great Famine_, the other notable point is the growth phase starting in the 70s. Visually, it can be seen as the start of a length growth phase in Ireland's population. According to the CSO (2004), 

    > "In the period 1971 to 2002 the population of Ireland has grown by almost one-third to reach nearly 4 million. The majority of this increase took place during two periods - the 1970s when the population grew by 390,000 and the period from 1996 to 2002 when the population increased by almost 300,000" (pg viii Ireland & the EU: Economic and Social Change 1973-2003)


### Next: 
I will create trend lines for the counties individually. 


In [6]:
# County Population Over Time 
fig = px.line(
    census_pop,
    x="CensusYear", 
    y="FMCombined",
    color="County",
    title="Population of Counties (1841-2022)",
    labels={"FMCombined": "Population", "CensusYear": "Year"},
    markers=True
)
fig.show()

The above plot of trend lines adds little to the analysis outside of flagging the increasing gap between population of Dublin compared to the rest of the country. Originally, my plan was to compare WDC counties versus the rest but based on this, I am going to separate Dublin out on its' own as it will skew the national figures disproportionately. I will then have four dataframes dedicated to national, Dublin, WDC and the rest. I will create a regional dataframe to compare the regions to national figures. I will achieve this by using the [Pandas groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [sum()](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.sum.html#pandas.core.groupby.DataFrameGroupBy.sum) and [reset_index()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html#pandas.DataFrame.reset_index) functions. 

In [7]:
# Updating Group Column Logic
census_pop["Group"] = census_pop["County"].apply(lambda x: "WDC" if x in wdc else "Dublin" if x == "Dublin" else ("State" if x == "State" else "Rest"))

# Creating Dublin Dataframe
df_dublin = census_pop[census_pop["Group"] == "Dublin"]

# Creating Regions Dataframe
df_regions = census_pop.groupby(["CensusYear", "Group"])["FMCombined"].sum().reset_index()




In [8]:
# Plotting Regional Trends
fig = px.line(
    df_regions,
    x="CensusYear",
    y="FMCombined",
    color="Group",
    title="Population Trends by Region (WDC, Dublin, Rest, State)",
    labels={"FMCombined": "Population", "CensusYear": "Year"},
    markers=True,
    height=600
)
fig.show()


The trend lines above are more useful now. Already, the population of the WDC stands out for appearing to have a greater decline and slower increase. I want to start applying some of the testing inspired by _The Lady Tasting Tea_ expirement. To do this, I am going to create a few new dataframes for transforming the data into wide format. This will facilitate calculating and adding a column for [percentage change](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html). I will start with the **census_pop** dataframe again and re-format it. I will use [Pandas pivot_table()](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html) function to do this. 

In [10]:
# Pivotting
census_pop_wide = census_pop.pivot_table(index="County", columns="CensusYear", values="FMCombined")

# Sorting columns chronilogically
census_pop_wide = census_pop_wide.sort_index(axis=1)

In [12]:
# Printing to check
census_pop_wide.head()

CensusYear,1841,1851,1861,1871,1881,1891,1901,1911,1926,1936,...,1979,1981,1986,1991,1996,2002,2006,2011,2016,2022
County,,,,,,,,,,,,,,,,,,,,,
Carlow,86228.0,68078.0,57137.0,51650.0,46568.0,40936.0,37748.0,36252.0,34476.0,34452.0,...,38668.0,39820.0,40988.0,40942.0,41616.0,46014.0,50349.0,54612.0,56932.0,61968.0
Cavan,243158.0,174071.0,153906.0,140735.0,129476.0,111917.0,97541.0,91173.0,82452.0,76670.0,...,53720.0,53855.0,53965.0,52796.0,52944.0,56546.0,64003.0,73183.0,76176.0,81704.0
Clare,286394.0,212440.0,166305.0,147864.0,141457.0,124483.0,112334.0,104232.0,95064.0,89879.0,...,84919.0,87567.0,91344.0,90918.0,94006.0,103277.0,110950.0,117196.0,118817.0,127938.0
Cork,854118.0,649308.0,544818.0,517076.0,495607.0,438432.0,404611.0,392104.0,365747.0,355957.0,...,396118.0,402465.0,412735.0,410369.0,420510.0,447829.0,481295.0,519032.0,542868.0,584156.0
Donegal,296448.0,255158.0,237395.0,218334.0,206035.0,185635.0,173722.0,168537.0,152508.0,142310.0,...,121941.0,125112.0,129664.0,128117.0,129994.0,137575.0,147264.0,161137.0,159192.0,167084.0
